# Paredes por píxel

> ¿El modelo encuentra las paredes, pero las corta distinto que la etiqueta? En cada plano junta todas las paredes en un mapa de píxeles, uno predicho y otro etiquetado, y los compara en FloorPlanCAD y en BLD-AR.

In [ ]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

In [ ]:
%pip install ultralytics==8.4.173 sahi==0.12.8

## Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import yaml
from IPython.display import Markdown, display
from PIL import Image, ImageDraw
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

## Rutas y parámetros

In [ ]:
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
BLDAR = REPO / "datasets/BLD-AR"
FLOORPLANCAD = REPO / "datasets/floor-plan-cad-yolo"
WEIGHTS = REPO / "weights/00-YOLO26m-SEG-1024/best.pt"

TILE = 1024
OVERLAP = 0.2
CONF = 0.25  # desde esta confianza una predicción cuenta como pared
DEVICE = "cuda:0"

# Clases de BLD-AR que se evalúan (docs/MAPEO_CLASES.md); las demás son zonas a ignorar
MAPPED = {"wall", "door", "window", "sliding_door", "stair", "elevator", "railing"}

assert WEIGHTS.exists(), f"No existen los pesos: {WEIGHTS}"

## Modelo

In [ ]:
model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(WEIGHTS),
    confidence_threshold=CONF, device=DEVICE, image_size=TILE,
)

## Mapas de pared y medidas

In [ ]:
def shape_points(shape):
    """Puntos de una forma LabelMe; los círculos se aproximan con 32 lados."""
    points = np.array(shape["points"], dtype=float)
    if shape["shape_type"] == "circle":
        center, edge = points
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        points = center + np.linalg.norm(edge - center) * np.stack([np.cos(angles), np.sin(angles)], axis=1)
    return points


def pixel_map(size, polygons):
    """Mapa del tamaño del plano, True dentro de los polígonos (listas x, y, x, y, ...)."""
    canvas = Image.new("1", size)
    draw = ImageDraw.Draw(canvas)
    for polygon in polygons:
        draw.polygon(polygon, fill=1)
    return np.array(canvas)


def predicted_walls(image_path):
    """Polígonos de wall que predice el modelo por tiles, como en la evaluación en BLD-AR."""
    result = get_sliced_prediction(
        str(image_path), model,
        slice_height=TILE, slice_width=TILE,
        overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
        perform_standard_pred=False, verbose=0,
    )
    return [polygon for prediction in result.to_coco_predictions()
            if prediction["category_name"] == "wall" for polygon in prediction["segmentation"]]


def overlap(predicted, labeled):
    """Coincidencia, pared encontrada y pared bien marcada de un plano."""
    both = (predicted & labeled).sum()
    well_marked = both / predicted.sum() if predicted.any() else np.nan
    return both / (predicted | labeled).sum(), both / labeled.sum(), well_marked

## FloorPlanCAD (test)

In [ ]:
names = yaml.safe_load((FLOORPLANCAD / "floor-plan-cad-yolo.yaml").read_text())["names"]
wall_id = next(i for i, name in names.items() if name == "wall")

floorplancad = []
for image_path in sorted((FLOORPLANCAD / "images/test").iterdir()):
    size = Image.open(image_path).size
    rows = [line.split() for line in (FLOORPLANCAD / "labels/test" / f"{image_path.stem}.txt").read_text().splitlines()]
    # Etiqueta YOLO: clase y puntos x, y entre 0 y 1
    walls = [(np.reshape(np.array(row[1:], dtype=float), (-1, 2)) * size).ravel().tolist()
             for row in rows if int(row[0]) == wall_id]
    if walls:
        floorplancad.append(overlap(pixel_map(size, predicted_walls(image_path)), pixel_map(size, walls)))

## BLD-AR

> Lo que el modelo predice como pared sobre las clases excluidas no cuenta, como en la evaluación.

In [ ]:
bldar = []
for json_path in sorted(BLDAR.glob("*.json")):
    data = json.loads(json_path.read_text())
    size = (data["imageWidth"], data["imageHeight"])
    walls = [shape_points(shape).ravel().tolist() for shape in data["shapes"] if shape["label"] == "wall"]
    ignored = [shape_points(shape).ravel().tolist() for shape in data["shapes"] if shape["label"] not in MAPPED]
    if walls:
        labeled = pixel_map(size, walls)
        predicted = pixel_map(size, predicted_walls(BLDAR / data["imagePath"]))
        predicted &= labeled | ~pixel_map(size, ignored)  # se conserva donde la etiqueta es pared
        bldar.append(overlap(predicted, labeled))

## Resultado

> Promedio por plano, en los planos con paredes etiquetadas. La hipótesis se confirma si la pared encontrada en BLD-AR llega al 80 % o más de la de FloorPlanCAD, y se descarta si queda debajo del 50 %. No se usa la coincidencia porque FloorPlanCAD tiene muros sin etiquetar que el modelo marca.

In [ ]:
results = {"FloorPlanCAD": floorplancad, "BLD-AR": bldar}
means = {name: np.nanmean(values, axis=0) for name, values in results.items()}
ratio = means["BLD-AR"] / means["FloorPlanCAD"]

table = "| | Planos | Coincidencia | Pared encontrada | Pared bien marcada |\n|---|---|---|---|---|\n"
for name, mean in means.items():
    table += f"| {name} | {len(results[name])} | {mean[0]:.3f} | {mean[1]:.3f} | {mean[2]:.3f} |\n"
table += f"| BLD-AR / FloorPlanCAD | | {ratio[0]:.0%} | {ratio[1]:.0%} | {ratio[2]:.0%} |"
display(Markdown(f"**{WEIGHTS.parent.name} · confianza {CONF}**\n\n" + table))